# OfflineMedia Portable - Google Colab

This notebook prepares a real NVIDIA GPU runtime, installs ComfyUI, downloads the verified Wan 2.1 assets, synchronizes the current official Wan workflows, and runs a real text-to-video smoke test.

**Evidence rule:** configuration is not a successful generation. The notebook only reports success after ComfyUI returns an output and the resulting video is validated.

In [ ]:
import os, sys, subprocess, time, json, shutil, urllib.request
from pathlib import Path

REPO = Path('/content/offlinemedia')
BRANCH = 'claude/scan-repo-chatgpt-review-6nljgh'
COMFY = Path('/content/ComfyUI')
print('Python:', sys.version.split()[0])
print('GPU check:')
subprocess.run(['nvidia-smi'], check=False)

In [ ]:
if not REPO.exists():
    subprocess.run(['git', 'clone', '-b', BRANCH, 'https://github.com/CAption11/offlinemedia.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'reset', '--hard', f'origin/{BRANCH}'], check=True)
os.chdir(REPO)
print('Repository:', REPO)

In [ ]:
# Install the repository's Portable dependencies and the Colab tooling.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt', 'huggingface_hub'], check=True)
if not COMFY.exists():
    subprocess.run(['git', 'clone', 'https://github.com/comfyanonymous/ComfyUI.git', str(COMFY)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(COMFY / 'requirements.txt')], check=True)
print('ComfyUI prepared:', COMFY)

In [ ]:
# Download the official Wan 2.1 T2V 1.3B assets used by the official ComfyUI example.
from huggingface_hub import hf_hub_download

HF_REPO = 'Comfy-Org/Wan_2.1_ComfyUI_repackaged'
assets = {
    'split_files/diffusion_models/wan2.1_t2v_1.3B_fp16.safetensors': COMFY / 'models/diffusion_models/wan2.1_t2v_1.3B_fp16.safetensors',
    'split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors': COMFY / 'models/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors',
    'split_files/vae/wan_2.1_vae.safetensors': COMFY / 'models/vae/wan_2.1_vae.safetensors',
}
for remote, local in assets.items():
    local.parent.mkdir(parents=True, exist_ok=True)
    if not local.exists():
        print('Downloading', remote)
        downloaded = hf_hub_download(HF_REPO, remote)
        shutil.copy2(downloaded, local)
    else:
        print('Already present:', local)
print('Wan T2V assets ready.')

In [ ]:
# Pull the current official Wan workflows. These are UI-format workflows;
# OfflineMedia's Workflow loader converts them using /object_info.
subprocess.run([sys.executable, 'portable/sync_official_workflows.py', '--output-dir', 'workflows/official'], check=True)
print(Path('workflows/official/text_to_video.json').read_text()[:120])

In [ ]:
# Start ComfyUI in the background.
log_path = Path('/content/comfyui.log')
log_file = log_path.open('w')
comfy_process = subprocess.Popen([sys.executable, str(COMFY / 'main.py'), '--listen', '127.0.0.1', '--port', '8188'], stdout=log_file, stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=2) as response:
            if response.status == 200:
                print('ComfyUI is ready.')
                break
    except Exception:
        time.sleep(2)
else:
    print(log_path.read_text(errors='replace')[-6000:])
    raise RuntimeError('ComfyUI did not become ready within 4 minutes.')

In [ ]:
# Run the shared OfflineMedia engine against the official T2V workflow.
# The official workflow emits WEBM/WEBP; the next cell converts the video to MP4.
from app.core.generation import GenerationRequest, GenerationType
from app.engines.comfyui_client import ComfyUIClient
from app.engines.comfyui_engine import ComfyUIEngine

client = ComfyUIClient(host='127.0.0.1', port=8188, timeout=30)
if not client.is_available():
    raise RuntimeError('ComfyUI is not reachable.')

request = GenerationRequest(
    generation_type=GenerationType.TEXT_TO_VIDEO,
    prompt='A small red ball rolling across a wooden table, natural lighting, smooth camera movement',
    negative_prompt='blurry, distorted, low quality, static image',
    width=832,
    height=480,
    frames=33,
    fps=16,
    seed=12345,
    output_dir=REPO / 'projects' / 'smoke_tests',
)
engine = ComfyUIEngine(client, REPO / 'workflows', REPO / 'projects' / 'smoke_tests')
# Use the official workflow directly so the test does not depend on a guessed graph.
from app.engines.workflow import Workflow
workflow = Workflow.load(REPO / 'workflows' / 'official' / 'text_to_video.json', client.object_info())
values = {'prompt': request.prompt, 'negative_prompt': request.negative_prompt, 'width': request.width, 'height': request.height, 'frames': request.frames, 'fps': request.fps, 'seed': request.seed}
workflow.auto_bind(values)
job_id = client.queue_prompt(workflow.to_dict())
history = client.wait_for_completion(job_id, timeout=3600)
items = client.output_items(history)
if not items:
    raise RuntimeError(f'ComfyUI completed but returned no outputs: {history}')
out_dir = REPO / 'projects' / 'smoke_tests' / job_id
downloaded = []
for index, item in enumerate(items, 1):
    suffix = Path(item['filename']).suffix or '.bin'
    destination = out_dir / f'comfy_output_{index:03d}{suffix}'
    downloaded.append(client.download_output(item, destination))
print('Job:', job_id)
for path in downloaded: print('Output:', path)

In [ ]:
# Validate and convert the first video output to MP4.
video_candidates = [p for p in downloaded if p.suffix.lower() in {'.webm', '.mp4', '.mkv', '.mov'}]
if not video_candidates:
    raise RuntimeError(f'No video output returned. Files: {downloaded}')
source = video_candidates[0]
mp4 = source.with_suffix('.mp4')
if source.suffix.lower() == '.mp4':
    mp4 = source
else:
    subprocess.run(['ffmpeg', '-y', '-i', str(source), '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-movflags', '+faststart', str(mp4)], check=True)
size = mp4.stat().st_size
if size <= 1024:
    raise RuntimeError(f'Generated MP4 is suspiciously small: {size} bytes')
probe = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration,size', '-of', 'json', str(mp4)], capture_output=True, text=True, check=True)
print('REAL GENERATION PASSED')
print('MP4:', mp4)
print(probe.stdout)

In [ ]:
from IPython.display import Video, display
display(Video(str(mp4), embed=True))

## Image-to-video

The official Wan 2.1 image-to-video example uses the 14B 480p model plus `clip_vision_h.safetensors`. It is intentionally not executed in this first smoke test. After T2V is proven, add those assets and run the same shared engine against `workflows/official/image_to_video.json`.